In [99]:
!pip -q install snowflake-connector-python pandas requests

In [100]:
import os
import requests
import pandas as pd
from datetime import date, timedelta
import snowflake.connector
from IPython.display import display

In [101]:
def geocode_city(city: str, country_code: str = "US", count: int = 1):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {"name": city, "count": count, "language": "en", "format": "json"}
    if country_code:
        params["country_code"] = country_code

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    if "results" not in data or len(data["results"]) == 0:
        raise ValueError(f"No geocoding results for city='{city}'")

    top = data["results"][0]
    return {
        "name": top.get("name"),
        "admin1": top.get("admin1"),
        "country": top.get("country"),
        "latitude": float(top["latitude"]),
        "longitude": float(top["longitude"]),
    }

CITY = "Seattle"
geo = geocode_city(CITY, country_code="US")
geo

{'name': 'Seattle',
 'admin1': 'Washington',
 'country': 'United States',
 'latitude': 47.60621,
 'longitude': -122.33207}

In [102]:
from google.colab import userdata

SF_USER = userdata.get("SNOWFLAKE_USER")
SF_PASSWORD = userdata.get("SNOWFLAKE_PASSWORD")
SF_ACCOUNT = userdata.get("SNOWFLAKE_ACCOUNT")
SF_WAREHOUSE = userdata.get("SNOWFLAKE_WAREHOUSE")
SF_DATABASE = userdata.get("SNOWFLAKE_DATABASE")
SF_SCHEMA = userdata.get("SNOWFLAKE_SCHEMA")
SF_ROLE = userdata.get("SNOWFLAKE_ROLE")

SF_TABLE = "WEATHER_DAILY"

TARGET_TABLE = f"{SF_DATABASE}.{SF_SCHEMA}.{SF_TABLE}"

assert SF_USER and SF_PASSWORD and SF_ACCOUNT and SF_WAREHOUSE and SF_DATABASE and SF_SCHEMA

In [116]:
def extract_last_60_days_weather(latitude: float, longitude: float, timezone: str = "America/Los_Angeles") -> pd.DataFrame:
    end_date = date.today() - timedelta(days=1)
    start_date = end_date - timedelta(days=59)
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,weather_code",
        "timezone": timezone,
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    if "daily" not in data or "time" not in data["daily"]:
        raise ValueError(f"Unexpected API response (missing daily/time). Response keys: {list(data.keys())}")
    dates = pd.Series(pd.to_datetime(data["daily"]["time"])).dt.date
    df = pd.DataFrame({
        "latitude": latitude,
        "longitude": longitude,
        "date": dates,
        "temp_max": data["daily"].get("temperature_2m_max", []),
        "temp_min": data["daily"].get("temperature_2m_min", []),
        "precipitation": data["daily"].get("precipitation_sum", []),
        "weather_code": data["daily"].get("weather_code", []),
    })
    return df
CITY = "Seattle"
geo = geocode_city(CITY, country_code="US")
display(pd.DataFrame([geo]))
df_weather = extract_last_60_days_weather(geo["latitude"], geo["longitude"])
display(df_weather.head(5))
display(pd.DataFrame([{
    "rows": len(df_weather),
    "min_date": df_weather["date"].min(),
    "max_date": df_weather["date"].max()
}]))

,name,admin1,country,latitude,longitude
0,Seattle,Washington,United States,47.60621,-122.33207


,latitude,longitude,date,temp_max,temp_min,precipitation,weather_code
0,47.60621,-122.33207,2025-12-28,5.2,-0.2,0.1,51
1,47.60621,-122.33207,2025-12-29,5.6,1.5,0.0,3
2,47.60621,-122.33207,2025-12-30,6.7,1.4,0.0,3
3,47.60621,-122.33207,2025-12-31,5.7,1.0,0.0,3
4,47.60621,-122.33207,2026-01-01,4.6,1.5,1.8,53


,rows,min_date,max_date
0,60,2025-12-28,2026-02-25


In [117]:
def _normalize_account(account: str) -> str:
    a = account.strip()
    if a.startswith("http://") or a.startswith("https://"):
        a = a.split("://", 1)[1]
    a = a.split("/", 1)[0]
    suffix = ".snowflakecomputing.com"
    if a.endswith(suffix):
        a = a[: -len(suffix)]
    return a

In [118]:
def return_snowflake_conn():
    conn = snowflake.connector.connect(
        user=user_id,
        password=password,
        account=account,
        warehouse=warehouse,
        database=database,
        schema=schema,
        role=role if role else None
    )
    return conn.cursor()

In [119]:
def extract(latitude, longitude):
    """Get the past 60 days of daily weather for coordinates."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "past_days": 60,
        "forecast_days": 0,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "weather_code"
        ],
        "timezone": "America/Los_Angeles"
    }

    response = requests.get(url, params=params, timeout=60)
    if response.status_code != 200:
        raise RuntimeError(f"API request failed: {response.status_code} - {response.text[:200]}")
    return response.json()

In [120]:
def transform(raw_data, latitude, longitude):
    if "daily" not in raw_data:
        raise ValueError("Missing 'daily' key in API response")

    daily = raw_data["daily"]

    df = pd.DataFrame({
        "latitude": latitude,
        "longitude": longitude,
        "date": pd.to_datetime(daily["time"]).date,
        "temp_max": daily["temperature_2m_max"],
        "temp_min": daily["temperature_2m_min"],
        "precipitation": daily["precipitation_sum"],
        "weather_code": daily["weather_code"],
    })

    return df

In [121]:
def load(con, target_table, records: pd.DataFrame):
    """
    Full refresh:
      - create table if not exists
      - delete all
      - insert all rows
    con is a Snowflake CURSOR (so con.execute works)
    """
    create_sql = f"""
    CREATE TABLE IF NOT EXISTS {target_table} (
        latitude      FLOAT,
        longitude     FLOAT,
        date          DATE,
        temp_max      FLOAT,
        temp_min      FLOAT,
        precipitation FLOAT,
        weather_code  INTEGER,
        PRIMARY KEY (latitude, longitude, date)
    );
    """
    delete_sql = f"DELETE FROM {target_table};"
    insert_sql = f"""
    INSERT INTO {target_table}
      (latitude, longitude, date, temp_max, temp_min, precipitation, weather_code)
    VALUES
      (%(latitude)s, %(longitude)s, %(date)s, %(temp_max)s, %(temp_min)s, %(precipitation)s, %(weather_code)s);
    """
    payload = records.to_dict(orient="records")
    try:
        con.execute("BEGIN;")
        con.execute(create_sql)
        con.execute(delete_sql)
        con.executemany(insert_sql, payload)
        con.execute("COMMIT;")
        print(f"Loaded {len(records)} records into {target_table}")
    except Exception as e:
        con.execute("ROLLBACK;")
        raise e

In [122]:
def ETL_pipeline(con, target_table, latitude, longitude):
    raw_data = extract(latitude, longitude)
    df = transform(raw_data, latitude, longitude)
    load(con, target_table, df)

In [123]:
def check_table_stats(con, table):
    con.execute(f"SELECT * FROM {table}")
    df = con.fetch_pandas_all()
    print(f"{len(df)} records in {table}")
    return df

In [124]:
LATITUDE = 47.6062
LONGITUDE = -122.3321

target_table = f"{SF_DATABASE}.{SF_SCHEMA}.WEATHER_DAILY"

snowflake_con = return_snowflake_conn()

ETL_pipeline(snowflake_con, target_table, LATITUDE, LONGITUDE)
check_table_stats(snowflake_con, target_table)

ETL_pipeline(snowflake_con, target_table, LATITUDE, LONGITUDE)
check_table_stats(snowflake_con, target_table)

snowflake_con.close()

Loaded 60 records into USER_DB_HIPPO.RAW.WEATHER_DAILY
60 records in USER_DB_HIPPO.RAW.WEATHER_DAILY
Loaded 60 records into USER_DB_HIPPO.RAW.WEATHER_DAILY
60 records in USER_DB_HIPPO.RAW.WEATHER_DAILY


True